# 01 — Environment Check

**Project:** Lightweight domain adaptation of small LLMs for chemistry using LoRA and QLoRA (MSc FYP).

**Purpose of this notebook:** verify that the Colab runtime is ready for PEFT experiments — GPU available, CUDA working, and the core ML libraries installed at compatible versions.

Run all cells top-to-bottom. If any cell fails, fix it before moving on to `02_*`.

## 1. Runtime: Python and OS

In [8]:
import sys, platform
print("Python   :", sys.version.split()[0])
print("Platform :", platform.platform())

Python   : 3.12.13
Platform : Linux-6.6.122+-x86_64-with-glibc2.35


## 2. GPU: `nvidia-smi`

If this prints `command not found`, go to **Runtime → Change runtime type → T4 GPU** (or better) and re-run.

In [9]:
!nvidia-smi

Thu Jun 18 11:59:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             29W /   70W |     959MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. PyTorch and CUDA

In [10]:
import torch
print("torch              :", torch.__version__)
print("CUDA available     :", torch.cuda.is_available())
print("CUDA version       :", torch.version.cuda)
if torch.cuda.is_available():
    print("Device count       :", torch.cuda.device_count())
    print("Device name        :", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"Total memory (GiB) : {props.total_memory / 1024**3:.2f}")

torch              : 2.11.0+cu128
CUDA available     : True
CUDA version       : 12.8
Device count       : 1
Device name        : Tesla T4
Total memory (GiB) : 14.56


## 4. Install PEFT stack

Pinned to versions known to work together on Colab as of mid-2026. Restart the runtime after this cell if prompted.

In [11]:
%pip install -q \
    transformers \
    peft \
    bitsandbytes \
    accelerate \
    datasets \
    trl \
    sentencepiece

## 5. Verify installed versions

In [12]:
import importlib
for pkg in ["transformers", "peft", "bitsandbytes", "accelerate", "datasets", "trl"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:14s} : {mod.__version__}")
    except Exception as e:
        print(f"{pkg:14s} : FAILED — {e}")

transformers   : 5.10.2
peft           : 0.19.1
bitsandbytes   : 0.49.2
accelerate     : 1.13.0
datasets       : 5.0.0
trl            : 1.6.0


## 6. Smoke test: load a tiny model in 4-bit

Confirms bitsandbytes + transformers + CUDA all cooperate. Uses TinyLlama (~1.1B) to stay fast.

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded.")
print("Memory footprint (MiB):", model.get_memory_footprint() / 1024**2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded.
Memory footprint (MiB): 712.176025390625


In [14]:
prompt = "Briefly, what is a catalyst in chemistry?"
inputs = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=60, do_sample=False)
print(tok.decode(out[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Briefly, what is a catalyst in chemistry?


## 7. Done

If every cell above ran without error, the environment is ready. Next: `02_*` — dataset selection / preparation.